In [3]:
# Important imports
from abc import ABC, abstractmethod
from typing import Optional, List, Type, Tuple, Dict
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.axes._axes import Axes
import torch
import torch.distributions as D
from torch.func import vmap, jacrev
import tqdm
import seaborn as sns
from sklearn.datasets import make_moons, make_circles

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Important Basic Functions
class Sampleable(ABC):
    """
    Distribution which can be sample from
    """
    @property
    @abstractmethod
    def dim(self) -> int:
        """
        Returns:
            -Dimensionality of the distribution
        """
        pass

    @abstractmethod
    def sample(self, num_samples: int) -> torch.Tensor:
        """
        Args:
            -num_samples : the desired number of samples
        Return:
            -samples: shape = (bs, dim)
        """
        pass


class Density(ABC):
    """
    Distribution with tractable density
    """
    @abstractmethod
    def log_density(self, x: torch.Tensor) -> torch.Tensor:
        """
        - Returns the log density at x
        Args:
            -x: shape = (bs, dim)
        Returns:
            -log_density: shape=(bs, 1)
        """
        pass


class Gaussian(torch.nn.Module, Sampleable, Density):
    """
    - Multivariate Gaussian distribution
    """
    def __init__(self, mean: torch.Tensor, cov: torch.Tensor):
        """
        mean : shape=(dim, )
        cov: shape=(dim, dim)
        """
        super().__init__()
        self.register_buffer("mean", mean)
        self.register_buffer("cov", cov)

    @property
    def dim(self) -> int:
        return self.mean.shape[0]

    @property
    def distribution(self):
        return D.MultivariateNormal(self.mean, self.cov, validate_args=False)

    def sample(self, num_samples: int) -> torch.Tensor:
        return self.distribution.sample((num_samples, ))

    def log_density(self, x: torch.Tensor):
        return self.distribution.log_prob(x).view(-1, 1)

    @classmethod
    def isotropic(cls, dim: int, std: float) -> "Gaussian":
        mean = torch.zeros(dim)
        cov = torch.eye(dim) * std **2
        return cls(mean, cov)

class GaussianMixture(torch.nn.Module, Sampleable, Density):
    """
    2D Mixture Gaussian model (Density and Sampleable). Wraper around torch.distributions.MixtureSameFamily.
    """
    def __init__(self,
                 means: torch.Tensor, # nmodes x data_dim
                 covs: torch.Tensor, # nmodes x data_dim x data_dim
                 weights: torch.Tensor # nmodes
                ):
        """
        means: shape=(nmodes, 2)
        covs: shape=(nmodes, 2, 2)
        weights: shape=(nmodes, 1)
        """
        super().__init__()
        self.nmodes = means.shape[0]
        self.register_buffer("means", means) 
        self.register_buffer("covs", covs) 
        self.register_buffer("weights", weights) 


    @property
    def dim(self) -> int:
        return self.means.shape[1]

    @property
    def distribution(self):
        return D.MixtureSameFamily(
            mixture_distribution=D.Categorical(probs=self.weights, validate_args=False),
            component_distribution=D.MultivariateNormal(
                loc=self.means,
                covariance_matrix=self.covs,
                validate_args=False
            ),
            validate_args=False,
        )

    def log_density(self. x: torch.Tensor) -> torch.Tensor:
        return self.distribution.log_prob(x).view(-1, 1)

    def sample(self, num_samples: int) -> torch.Tensor:
        return self.distribution.sample(torch.Size((num_samples,)))


    @classmethod
    def random_2D(
        cls, nmodes: int, std: float, scale: float = 10.0, x_offset: float=0.0, seed=0.0
    ) -> "GaussianMixture":
        torch.manual_seed(seed)
        means = (torch.rand(nmodes, 2) - 0.5) * scale + x_offset * torch.Tensor([1.0, 0.0])
        covs = torch.diag_embed(torch.ones(nmodes, 2)) * std**2
        weights = torch.ones(nmodes)
        return cls(means, covs, weights)

    @classmethod
    def symmetric_2D(
        cls, nmodes: int, std: float, scale: float=10.0, x_offset: float=0.0
    ) -> "GaussianMixture":
        angles = torch.linspace(0, 2*np.pi, nmodes + 1)[: nmodes]
        means = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * scale + torch.Tensor([1.0, 0.0]) * x_offset
        covs = torch.diag_embed(torch.ones(nmodes, 2) * std **2)
        weights = torch.ones(nmodes) / nmodes
        return cls(measn, covs, weights)


        



    
    
